# 企业RAG — 多模态PDF处理
## 文本 / 表格 / 图片 分类处理

```
PDF页面
├── 文本块  ──→ pdfplumber 提取 ──→ 直接分块 ──→ Embedding
├── 表格    ──→ pdfplumber 结构化 ──→ Markdown表格 ──→ Embedding
└── 图片    ──→ PyMuPDF 提取 ──→ OCR / 视觉模型描述 ──→ Embedding
```

每种类型有不同的提取策略和元数据，最终统一写入 ChromaDB。

## Step 0：安装依赖

In [ ]:
%pip install -q \
    pdfplumber \
    pymupdf \
    pytesseract \
    Pillow \
    matplotlib \
    reportlab \
    langchain \
    langchain-community \
    langchain-chroma \
    langchain-huggingface \
    sentence-transformers \
    chromadb \
    python-dotenv
print('依赖安装完成')

## Step 1：生成富内容测试PDF（含文本 + 表格 + 图片）

In [ ]:
import io
import os
import matplotlib
matplotlib.use("Agg")  # 非交互式后端，避免弹窗
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image as RLImage, PageBreak, HRFlowable
)

PDF_PATH = "multimodal_report.pdf"
W, H = A4
styles = getSampleStyleSheet()
body  = styles["BodyText"]
h1    = styles["Heading1"]
h2    = styles["Heading2"]

# ── 辅助：生成 matplotlib 图表并返回 BytesIO ──────────────────
def make_bar_chart() -> io.BytesIO:
    fig, ax = plt.subplots(figsize=(6, 3))
    quarters = ["Q1", "Q2", "Q3", "Q4"]
    revenue  = [120, 145, 162, 198]
    cost     = [80, 90, 95, 110]
    x = np.arange(len(quarters))
    ax.bar(x - 0.2, revenue, 0.4, label="营收（百万）", color="#4C72B0")
    ax.bar(x + 0.2, cost,    0.4, label="成本（百万）", color="#DD8452")
    ax.set_xticks(x); ax.set_xticklabels(quarters)
    ax.set_title("2025年度季度财务概览")
    ax.legend()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    return buf

def make_pie_chart() -> io.BytesIO:
    fig, ax = plt.subplots(figsize=(5, 4))
    labels  = ["智能客服", "风控系统", "合规审查", "研究分析", "其他"]
    sizes   = [35, 25, 20, 12, 8]
    ax.pie(sizes, labels=labels, autopct="%1.0f%%",
           colors=["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"])
    ax.set_title("AI应用场景投入分布")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    return buf

# ── 构建PDF内容 ───────────────────────────────────────────────
story = []

# 封面
story += [
    Spacer(1, 3*cm),
    Paragraph("企业人工智能战略报告 2025", ParagraphStyle(
        "cover", parent=h1, fontSize=22, spaceAfter=20, alignment=1)),
    Paragraph("金融科技部 · 内部保密文件", ParagraphStyle(
        "subtitle", parent=body, fontSize=12, alignment=1, textColor=colors.grey)),
    PageBreak(),
]

# 第1章 — 纯文本
story += [
    Paragraph("第一章 执行摘要", h1),
    Paragraph(
        "本报告系统梳理了本行2025年人工智能落地现状，评估了大语言模型（LLM）与检索增强生成"
        "（RAG）技术在各核心业务线的应用成效。研究表明，将企业内部知识库与生成式AI结合，能够"
        "显著提升员工效率并降低合规风险。", body),
    Spacer(1, 0.3*cm),
    Paragraph(
        "核心结论：智能客服场景中，RAG系统将人工接线率降低38%；合规审查自动化程度从12%提升至"
        "67%；研究报告生成周期从5天压缩至1.5天。三大场景综合ROI预计在18个月内转正。", body),
    PageBreak(),
]

# 第2章 — 包含数据表格
story += [
    Paragraph("第二章 业务场景效益分析", h1),
    Paragraph("下表汇总了各AI应用场景的关键指标对比（实施前 vs 实施后）：", body),
    Spacer(1, 0.4*cm),
]

table_data_1 = [
    ["应用场景",      "实施前指标",   "实施后指标",   "提升幅度",  "年化节省（万元）"],
    ["智能客服",      "人工接线100%", "人工接线62%",  "↑ 38%",    "1,200"],
    ["合规文件审查",  "自动化12%",    "自动化67%",    "↑ 55%",    "850"],
    ["研究报告生成",  "5个工作日",    "1.5个工作日",  "↓ 70%",    "320"],
    ["风控模型解释",  "人工撰写100%", "AI辅助80%",   "↑ 80%",    "480"],
    ["IT知识库问答",  "人工响应",     "自助解决率71%","↑ 71%",    "210"],
]

t1 = Table(table_data_1, colWidths=[3.8*cm, 3.2*cm, 3.2*cm, 2.5*cm, 3.5*cm])
t1.setStyle(TableStyle([
    ("BACKGROUND",   (0,0), (-1,0), colors.HexColor("#2C5F8A")),
    ("TEXTCOLOR",    (0,0), (-1,0), colors.white),
    ("FONTNAME",     (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE",     (0,0), (-1,-1), 8),
    ("ROWBACKGROUNDS",(0,1),(-1,-1), [colors.HexColor("#F0F4F8"), colors.white]),
    ("GRID",         (0,0), (-1,-1), 0.5, colors.HexColor("#AAAAAA")),
    ("ALIGN",        (0,0), (-1,-1), "CENTER"),
    ("VALIGN",       (0,0), (-1,-1), "MIDDLE"),
    ("TOPPADDING",   (0,0), (-1,-1), 4),
    ("BOTTOMPADDING",(0,0), (-1,-1), 4),
]))
story += [t1, PageBreak()]

# 第3章 — 包含柱状图图片
bar_buf = make_bar_chart()
story += [
    Paragraph("第三章 财务数据", h1),
    Paragraph(
        "图1展示了2025年各季度AI相关项目的营收贡献与成本投入对比。"
        "可以看出，Q3起营收增速明显超过成本增速，规模效应开始显现。", body),
    Spacer(1, 0.5*cm),
    RLImage(bar_buf, width=14*cm, height=7*cm),
    Spacer(1, 0.3*cm),
    Paragraph("图1：2025年季度财务对比（单位：百万元）",
              ParagraphStyle("caption", parent=body, fontSize=9,
                             alignment=1, textColor=colors.grey)),
    PageBreak(),
]

# 第4章 — 技术选型表格（多列）
story += [
    Paragraph("第四章 技术选型对比", h1),
    Paragraph("下表对比了主流向量数据库的核心能力，供架构选型参考：", body),
    Spacer(1, 0.4*cm),
]

table_data_2 = [
    ["数据库",   "部署方式",    "扩展性", "混合检索", "安全级别", "推荐场景"],
    ["Chroma",   "本地/私有云", "中等",   "否",       "高",       "POC/行内私有"],
    ["Weaviate", "云原生/自托管","高",    "是",       "高",       "生产规模"],
    ["Pinecone", "全托管SaaS",  "极高",   "是",       "中",       "快速上线"],
    ["Milvus",   "私有云",      "极高",   "是",       "极高",     "大规模私有"],
    ["PGVector", "PostgreSQL扩展","中",   "否",       "极高",     "已有PG基础"],
]

t2 = Table(table_data_2, colWidths=[2.5*cm, 3.5*cm, 2*cm, 2.5*cm, 2.5*cm, 3.5*cm])
t2.setStyle(TableStyle([
    ("BACKGROUND",   (0,0), (-1,0), colors.HexColor("#1A6B3A")),
    ("TEXTCOLOR",    (0,0), (-1,0), colors.white),
    ("FONTNAME",     (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE",     (0,0), (-1,-1), 8),
    ("ROWBACKGROUNDS",(0,1),(-1,-1), [colors.HexColor("#F0FFF4"), colors.white]),
    ("GRID",         (0,0), (-1,-1), 0.5, colors.HexColor("#AAAAAA")),
    ("ALIGN",        (0,0), (-1,-1), "CENTER"),
    ("VALIGN",       (0,0), (-1,-1), "MIDDLE"),
    ("TOPPADDING",   (0,0), (-1,-1), 5),
    ("BOTTOMPADDING",(0,0), (-1,-1), 5),
]))
story += [t2, PageBreak()]

# 第5章 — 饼图 + 文字混排
pie_buf = make_pie_chart()
story += [
    Paragraph("第五章 资源投入分配", h1),
    Paragraph(
        "图2展示了全行AI预算在五大应用场景的分配比例。智能客服占比最高（35%），"
        "主要因为该场景用户触达面广、数据积累充分，投资回报率最为确定。", body),
    Spacer(1, 0.5*cm),
    RLImage(pie_buf, width=10*cm, height=8*cm),
    Spacer(1, 0.3*cm),
    Paragraph("图2：AI场景预算分配（2025年度）",
              ParagraphStyle("caption", parent=body, fontSize=9,
                             alignment=1, textColor=colors.grey)),
    PageBreak(),
]

# 第6章 — 嵌入模型对比表格
story += [
    Paragraph("第六章 嵌入模型评测", h1),
    Paragraph(
        "我们在行内金融文档测试集上对主流嵌入模型进行了基准测试，"
        "评估维度包括检索准确率（MRR@10）、推理速度和内存占用：", body),
    Spacer(1, 0.4*cm),
]

table_data_3 = [
    ["模型",                      "维度",  "MRR@10（中文）","推理速度",    "内存",  "能否离线"],
    ["all-MiniLM-L6-v2",          "384",   "0.71",          "极快",        "90MB",  "是"],
    ["text2vec-large-chinese",    "1024",  "0.84",          "中等",        "650MB", "是"],
    ["BGE-M3",                    "1024",  "0.89",          "中等",        "580MB", "是"],
    ["OpenAI text-embedding-3-large","3072","0.93",         "API依赖",     "N/A",   "否"],
    ["bge-large-zh-v1.5",         "1024",  "0.87",          "中等",        "670MB", "是"],
]

t3 = Table(table_data_3, colWidths=[5*cm, 1.5*cm, 3.2*cm, 2.5*cm, 1.8*cm, 2*cm])
t3.setStyle(TableStyle([
    ("BACKGROUND",   (0,0), (-1,0), colors.HexColor("#6B2C8A")),
    ("TEXTCOLOR",    (0,0), (-1,0), colors.white),
    ("FONTNAME",     (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE",     (0,0), (-1,-1), 8),
    ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.HexColor("#F8F0FF"), colors.white]),
    ("GRID",         (0,0), (-1,-1), 0.5, colors.HexColor("#AAAAAA")),
    ("ALIGN",        (0,0), (-1,-1), "CENTER"),
    ("VALIGN",       (0,0), (-1,-1), "MIDDLE"),
    ("TOPPADDING",   (0,0), (-1,-1), 5),
    ("BOTTOMPADDING",(0,0), (-1,-1), 5),
]))
story += [t3, PageBreak()]

# 结论
story += [
    Paragraph("第七章 结论与建议", h1),
    Paragraph(
        "综合各章分析，本行AI知识库建设应遵循以下原则：\n"
        "1. 数据主权优先：所有向量和原始文档存储于行内私有云。\n"
        "2. 模型选型差异化：中文场景优先BGE-M3，资源受限场景用MiniLM。\n"
        "3. 分场景建库：智能客服、合规、研究分析各建独立Collection。\n"
        "4. 持续运营：建立文档更新流水线，保证知识库的时效性。", body),
    Spacer(1, 0.5*cm),
    HRFlowable(width="100%", thickness=1, color=colors.grey),
    Spacer(1, 0.3*cm),
    Paragraph("本文件为内部保密文件，未经授权不得外传。",
              ParagraphStyle("footer", parent=body, fontSize=8,
                             alignment=1, textColor=colors.grey)),
]

doc = SimpleDocTemplate(PDF_PATH, pagesize=A4,
                        leftMargin=2*cm, rightMargin=2*cm,
                        topMargin=2*cm, bottomMargin=2*cm)
doc.build(story)

size_kb = os.path.getsize(PDF_PATH) / 1024
print(f"测试PDF已生成: {PDF_PATH}  ({size_kb:.1f} KB)")
print("PDF包含：封面 + 文本页 + 3个数据表格 + 2张图表")

---
## ① 文本处理
使用 `pdfplumber` 提取带坐标信息的文本，并按语义分块。

In [ ]:
import pdfplumber
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

text_docs = []   # 最终文本Document列表

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", "。", "；", ".", " ", ""],
    add_start_index=True,
)

with pdfplumber.open(PDF_PATH) as pdf:
    print(f"总页数: {len(pdf.pages)}")
    for page_num, page in enumerate(pdf.pages):
        # 1. 提取纯文本（pdfplumber自动处理换行）
        raw_text = page.extract_text()
        if not raw_text or len(raw_text.strip()) < 30:
            print(f"  第{page_num+1}页: 文本过少，跳过")
            continue
        
        # 2. 构造Document并记录元数据
        page_doc = Document(
            page_content=raw_text,
            metadata={
                "source":      PDF_PATH,
                "page":        page_num,
                "content_type": "text",
                "page_width":  page.width,
                "page_height": page.height,
            }
        )
        
        # 3. 分块
        chunks = splitter.split_documents([page_doc])
        text_docs.extend(chunks)
        print(f"  第{page_num+1}页: {len(raw_text)}字符 → {len(chunks)} chunks")

print(f"\n文本类型总计: {len(text_docs)} 个Document")

---
## ② 表格处理

`pdfplumber` 能自动识别PDF中的表格边框并提取结构化行列数据。  
关键思路：**将表格转成 Markdown 格式**，让LLM能正确理解行列关系。

```
原始表格（二维列表）  →  Markdown表格字符串  →  Embedding
| 场景 | 指标 | 提升 |
|------|------|------|
| 客服 | 38%  | ...  |
```

In [ ]:
def table_to_markdown(table: list[list]) -> str:
    """将 pdfplumber 提取的二维列表转为 Markdown 表格字符串"""
    if not table or len(table) < 2:
        return ""
    
    # 清理None值
    cleaned = []
    for row in table:
        cleaned.append([str(cell).strip() if cell is not None else "" for cell in row])
    
    # 计算每列最大宽度（美化对齐）
    n_cols = max(len(r) for r in cleaned)
    col_widths = [0] * n_cols
    for row in cleaned:
        for i, cell in enumerate(row):
            if i < n_cols:
                col_widths[i] = max(col_widths[i], len(cell))
    col_widths = [max(w, 3) for w in col_widths]
    
    lines = []
    for idx, row in enumerate(cleaned):
        # 补齐列数
        padded = row + [""] * (n_cols - len(row))
        cells  = [f" {c.ljust(col_widths[i])} " for i, c in enumerate(padded)]
        lines.append("|" + "|".join(cells) + "|")
        # 表头分隔线
        if idx == 0:
            sep = [":"+"-"*col_widths[i]+":" for i in range(n_cols)]
            lines.append("|" + "|".join(sep) + "|")
    
    return "\n".join(lines)


table_docs = []   # 最终表格Document列表

with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if not tables:
            continue
        
        for tbl_idx, table in enumerate(tables):
            md_table = table_to_markdown(table)
            if len(md_table.strip()) < 20:
                continue
            
            # 获取表格在页面上的位置（bounding box）
            tbl_settings = page.find_tables()
            bbox = tbl_settings[tbl_idx].bbox if tbl_idx < len(tbl_settings) else None
            
            doc = Document(
                page_content=md_table,
                metadata={
                    "source":       PDF_PATH,
                    "page":         page_num,
                    "content_type": "table",
                    "table_index":  tbl_idx,
                    "bbox":         str(bbox),   # (x0, y0, x1, y1)
                    "rows":         len(table),
                    "cols":         len(table[0]) if table else 0,
                }
            )
            table_docs.append(doc)
            print(f"第{page_num+1}页 表格{tbl_idx+1}: {len(table)}行×{len(table[0])}列 → Markdown {len(md_table)}字符")
            print(md_table[:300])
            print()

print(f"表格类型总计: {len(table_docs)} 个Document")

---
## ③ 图片处理

使用 `PyMuPDF (fitz)` 提取PDF中嵌入的光栅图像（PNG/JPEG）。

### 图片处理的三条路径：
| 路径 | 工具 | 适合场景 | 效果 |
|------|------|----------|------|
| OCR文字识别 | `pytesseract` | 扫描件、含文字的截图 | 提取图内文字 |
| 视觉模型描述 | GPT-4V / Claude Vision | 图表、流程图 | 生成自然语言描述 |
| 元数据占位 | 直接记录图片位置 | 纯装饰图 | 最轻量，仅保留引用 |

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import io

IMG_DIR = "extracted_images"
os.makedirs(IMG_DIR, exist_ok=True)

image_docs   = []   # 最终图片Document列表
image_paths  = []   # 保存的图片文件路径

pdf_fitz = fitz.open(PDF_PATH)

for page_num in range(len(pdf_fitz)):
    page = pdf_fitz[page_num]
    image_list = page.get_images(full=True)
    
    if not image_list:
        continue
    
    for img_idx, img_info in enumerate(image_list):
        xref     = img_info[0]   # 图片在PDF内部的对象编号
        base_img = pdf_fitz.extract_image(xref)
        img_bytes  = base_img["image"]
        img_ext    = base_img["ext"]   # 'png' 或 'jpeg'
        img_width  = base_img["width"]
        img_height = base_img["height"]
        
        # 过滤过小的图（通常是装饰线、图标）
        if img_width < 50 or img_height < 50:
            continue
        
        # 保存图片文件
        img_filename = f"{IMG_DIR}/page{page_num+1}_img{img_idx+1}.{img_ext}"
        with open(img_filename, "wb") as f:
            f.write(img_bytes)
        image_paths.append(img_filename)
        
        print(f"第{page_num+1}页 图片{img_idx+1}: {img_width}×{img_height}px  → {img_filename}")
        
        # 创建Document（内容先留占位，Step后续用OCR或视觉模型填充）
        doc = Document(
            page_content=f"[图片占位] 来源：{PDF_PATH} 第{page_num+1}页 图{img_idx+1}",
            metadata={
                "source":       PDF_PATH,
                "page":         page_num,
                "content_type": "image",
                "image_path":   img_filename,
                "img_width":    img_width,
                "img_height":   img_height,
                "img_format":   img_ext,
            }
        )
        image_docs.append(doc)

pdf_fitz.close()
print(f"\n图片类型总计: {len(image_docs)} 张（已保存到 {IMG_DIR}/）")

### ③a 路径一：OCR 提取图内文字（适合扫描件）

> **前提**：需要安装 Tesseract OCR 引擎  
> Windows: https://github.com/UB-Mannheim/tesseract/wiki  
> Ubuntu: `apt install tesseract-ocr tesseract-ocr-chi-sim`

In [ ]:
import pytesseract
from PIL import Image

# Windows 用户需指定 Tesseract 路径（如未加入PATH）
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def ocr_image(image_path: str) -> str:
    """对图片进行OCR，返回识别文字。支持中英文混合。"""
    img = Image.open(image_path)
    # lang='chi_sim+eng'：同时识别简体中文和英文
    try:
        text = pytesseract.image_to_string(img, lang="chi_sim+eng",
                                           config="--psm 6")
        return text.strip()
    except Exception as e:
        return f"OCR失败: {e}"

# 对所有提取出的图片进行OCR
ocr_docs = []
for i, (img_path, placeholder_doc) in enumerate(zip(image_paths, image_docs)):
    ocr_text = ocr_image(img_path)
    if len(ocr_text) < 10:
        print(f"图片{i+1} ({img_path}): OCR未识别到有效文字（可能是纯图表）")
        continue
    
    doc = Document(
        page_content=f"[图片OCR内容] {ocr_text}",
        metadata={**placeholder_doc.metadata, "extract_method": "ocr"}
    )
    ocr_docs.append(doc)
    print(f"图片{i+1}: OCR结果 → {ocr_text[:150]}")

print(f"\nOCR成功: {len(ocr_docs)}/{len(image_docs)} 张图片")

### ③b 路径二：视觉模型描述图表（适合图表/流程图）

对于柱状图、饼图等OCR无法理解语义的图表，需要调用多模态LLM（GPT-4o / Claude）生成描述。

In [ ]:
import base64
from dotenv import load_dotenv
load_dotenv()

# ── 显示已提取的图片，人工确认内容 ───────────────────────────
from IPython.display import display, Image as IPImage
for img_path in image_paths:
    print(f"图片: {img_path}")
    display(IPImage(img_path, width=400))

def describe_image_with_claude(image_path: str) -> str:
    """
    调用 Anthropic Claude Vision API 描述图表内容。
    需要在 .env 中设置 ANTHROPIC_API_KEY
    """
    import anthropic
    
    with open(image_path, "rb") as f:
        img_data = base64.standard_b64encode(f.read()).decode("utf-8")
    
    ext = image_path.rsplit(".", 1)[-1].lower()
    media_type_map = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg"}
    media_type = media_type_map.get(ext, "image/png")
    
    client = anthropic.Anthropic()
    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=512,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image",
                     "source": {"type": "base64",
                                "media_type": media_type,
                                "data": img_data}},
                    {"type": "text",
                     "text": "请详细描述这张图表的内容，包括：图表类型、坐标轴含义、关键数据点和主要结论。用中文回答。"}
                ],
            }
        ],
    )
    return message.content[0].text


# 示例：对第一张图片调用视觉模型（需要ANTHROPIC_API_KEY）
vision_docs = []
if image_paths and os.getenv("ANTHROPIC_API_KEY"):
    for i, (img_path, placeholder_doc) in enumerate(zip(image_paths, image_docs)):
        print(f"正在描述图片: {img_path}")
        description = describe_image_with_claude(img_path)
        doc = Document(
            page_content=f"[图表描述] {description}",
            metadata={**placeholder_doc.metadata, "extract_method": "vision_model"}
        )
        vision_docs.append(doc)
        print(f"描述: {description[:200]}\n")
else:
    print("提示: 未设置ANTHROPIC_API_KEY，视觉模型描述已跳过")
    print("请在 .env 文件中设置 ANTHROPIC_API_KEY=sk-ant-xxx 后重新运行")
    # ── 演示用途：用规则生成占位描述 ────────────────────────
    for i, (img_path, placeholder_doc) in enumerate(zip(image_paths, image_docs)):
        w = placeholder_doc.metadata["img_width"]
        h = placeholder_doc.metadata["img_height"]
        desc = (f"来自第{placeholder_doc.metadata['page']+1}页的图表图片，"
                f"尺寸{w}×{h}像素。【建议配置视觉模型API以获取真实描述】")
        doc = Document(
            page_content=f"[图表描述] {desc}",
            metadata={**placeholder_doc.metadata, "extract_method": "placeholder"}
        )
        vision_docs.append(doc)

print(f"视觉描述文档: {len(vision_docs)} 个")

---
## ④ 汇总统计 — 三类内容分布

In [ ]:
# 合并所有类型（表格和图片用 vision_docs）
all_docs = text_docs + table_docs + vision_docs

from collections import Counter
type_counts = Counter(d.metadata["content_type"] for d in all_docs)

print("═" * 50)
print("  多模态PDF解析统计")
print("═" * 50)
print(f"  文本 chunks : {type_counts.get('text',  0):4d} 个")
print(f"  表格 docs   : {type_counts.get('table', 0):4d} 个")
print(f"  图片 docs   : {type_counts.get('image', 0):4d} 个")
print(f"  ──────────────────")
print(f"  合计        : {len(all_docs):4d} 个")
print("═" * 50)
print()

# 各类型示例
print("── 文本示例 ──")
if text_docs:
    print(text_docs[0].page_content[:200])
print()
print("── 表格示例 ──")
if table_docs:
    print(table_docs[0].page_content[:400])
print()
print("── 图片示例 ──")
if vision_docs:
    print(vision_docs[0].page_content[:200])

---
## ⑤ 向量化并写入 ChromaDB（分Collection存储）

三类内容分别存入独立 Collection，方便按类型检索或混合检索。

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_db"

print("加载嵌入模型...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("嵌入模型就绪")

stores = {}

# 文本 Collection
if text_docs:
    stores["text"] = Chroma.from_documents(
        text_docs, embeddings,
        collection_name="mm_text",
        persist_directory=CHROMA_DIR
    )
    print(f"mm_text   写入完成: {stores['text']._collection.count()} 条")

# 表格 Collection
if table_docs:
    stores["table"] = Chroma.from_documents(
        table_docs, embeddings,
        collection_name="mm_table",
        persist_directory=CHROMA_DIR
    )
    print(f"mm_table  写入完成: {stores['table']._collection.count()} 条")

# 图片 Collection
if vision_docs:
    stores["image"] = Chroma.from_documents(
        vision_docs, embeddings,
        collection_name="mm_image",
        persist_directory=CHROMA_DIR
    )
    print(f"mm_image  写入完成: {stores['image']._collection.count()} 条")

print("\n所有类型写入ChromaDB完成")

---
## ⑥ 混合检索测试（文本 + 表格 + 图片）

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def multimodal_search(query: str, k_per_type: int = 3, final_k: int = 5):
    """跨文本/表格/图片三个Collection检索，Cross-Encoder统一精排"""
    candidates = []
    
    # 从每个Collection召回
    for ctype, store in stores.items():
        results = store.similarity_search(query, k=k_per_type)
        for doc in results:
            doc.metadata["_collection"] = ctype  # 记录来源类型
        candidates.extend(results)
    
    if not candidates:
        print("未找到相关内容")
        return []
    
    # Cross-Encoder 精排
    pairs  = [(query, d.page_content) for d in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    
    print(f"查询: {query!r}")
    print(f"召回: {len(candidates)} 个（文本+表格+图片）→ 精排 Top-{final_k}")
    print("─" * 65)
    for rank, (score, doc) in enumerate(ranked[:final_k], 1):
        ctype    = doc.metadata.get("_collection", "?")
        page_num = doc.metadata.get("page", "?") 
        icon = {"text": "📄", "table": "📊", "image": "🖼️"}.get(ctype, "?")
        print(f"[{rank}] {icon} {ctype:6s} | 精排分:{score:6.3f} | 第{page_num+1 if isinstance(page_num,int) else page_num}页")
        print(f"      {doc.page_content[:250]}")
        print()
    
    return [doc for _, doc in ranked[:final_k]]


# 测试查询
results = multimodal_search("各业务场景AI实施效果提升了多少")

In [ ]:
results2 = multimodal_search("哪个数据库支持混合检索")

In [ ]:
results3 = multimodal_search("2025年季度财务营收成本对比图")

---
## 总结：三类内容的处理差异

| 内容类型 | 提取工具 | 处理方式 | 入库格式 | 关键元数据 |
|----------|----------|----------|----------|------------|
| **文本** | `pdfplumber.extract_text()` | RecursiveTextSplitter分块 | 纯文本字符串 | page, start_index |
| **表格** | `pdfplumber.extract_tables()` | 二维列表→Markdown表格 | Markdown字符串 | page, rows, cols, bbox |
| **图片** | `PyMuPDF.get_images()` | OCR 或 视觉模型描述 | 描述文本字符串 | page, image_path, 尺寸 |

### 为什么表格要转成 Markdown？
- 保留行列关系，LLM能理解「第3行第2列是什么」
- 向量模型对结构化文本有更好的语义捕获
- 避免原始CSV中分隔符干扰Embedding

### 为什么图片需要视觉模型？
- 纯Embedding无法理解像素内容
- 将图表转成自然语言描述后才能参与向量检索
- OCR只适合扫描文字，不适合可视化图表